# Ringer Dataset Example

This notebook shows how to generate a small fake parquet dataset, write it to disk, and load it with `RingerParquetDataset`.

# Importing Packages

In [1]:
from pathlib import Path
import sys
import os
# Adds repo root to sys.path so that we can import the neuralnet package
repo_path = Path(os.getcwd()).parent.parent.absolute()
sys.path.append(str(repo_path))

import polars as pl

from neuralnet.bins import AbsoluteVariableBin
from neuralnet.datasets.ringer import RingerParquetDataset, generate_ringer_dataset_dfs

# Generating Fake Data

In [2]:
dataset_dir = Path('dataset')
dataset_dir.mkdir(exist_ok=True)
et_bins = [
    dict(low=0.0, high=50000.0, closed="left"),
    dict(low=50000.0, high=100000.0, closed="left"),
]

eta_bins = [
    dict(low=0.0, high=0.8, closed="left"),
    dict(low=0.8, high=1.37, closed="left"),
]

data_df, kfold_df = generate_ringer_dataset_dfs(
    et_bins=et_bins,
    eta_bins=eta_bins,
    samples_per_bin=10000,
    n_folds=3,
    random_state=42,
)

In [3]:
data_df.write_parquet(dataset_dir / "electron_ringer.parquet")
data_df

id,rings,et,eta
u64,list[f32],f32,f32
0,"[-2.705197, 9.096173, … -7.955918]",48871.601562,0.797343
1,"[-9.522344, 2.591035, … 5.646278]",26096.707031,0.06353
2,"[-2.603605, 8.92839, … -7.899292]",13165.077148,0.617038
3,"[-9.464378, 2.82857, … 5.509369]",7989.174316,0.683489
4,"[-2.379803, 8.889713, … -7.701864]",368.296509,0.296633
…,…,…,…
39995,"[-2.510789, 9.011859, … -7.815817]",57667.445312,1.19812
39996,"[-2.55449, 9.052663, … -7.774921]",98959.210938,0.845549
39997,"[-9.287633, 2.760057, … 5.751782]",67914.570312,1.180212


In [4]:
kfold_df.write_parquet(dataset_dir / 'kfold_bins.parquet')
kfold_df

id,label,fold
i64,bool,i64
0,false,1
1,true,0
2,false,0
3,true,2
4,false,1
…,…,…
39995,false,2
39996,false,2
39997,true,1


# Reading with the RingerParquetDataset

To load the dataset you have to define what kind of et and eta bins you want to load. The data table represents the table with the rings, et and eta data and the kfold_table represent the table that contains the labels and the fold identification for each event

In [5]:
et_filter = AbsoluteVariableBin(var_name="et", low=0.0, high=100000.0, closed="left")
eta_filter = AbsoluteVariableBin(var_name="eta", low=0.0, high=1.37, closed="left")

dataset = RingerParquetDataset(
    dataset_dir=dataset_dir,
    data_table="electron_ringer",
    rings_col="rings",
    kfold_table="kfold_bins",
    fold_col="fold",
    fold=0,
    et_bin={
        'var_name': 'et',
        'low': 0.0,
        'high': 50000.0,
        'closed': 'left'
    },
    eta_bin={
        'var_name': 'eta',
        'low': 0.0,
        'high': 0.8,
        'closed': 'left'
    },
)
dataset

RingerParquetDataset(dataset_dir=PosixPath('dataset'), data_table='electron_ringer', rings_col='rings', kfold_table='kfold_bins', label_col='label', fold_col='fold', fold=0, et_bin=VariableBin(var_name='et', low=0.0, high=50000.0, closed='left'), eta_bin=AbsoluteVariableBin(var_name='eta', low=0.0, high=0.8, closed='left'))

In [6]:
train_df = dataset.train_df()   # Only training samples
val_df = dataset.val_df()   # Only validation samples
test_df = dataset.test_df()     # Test + Validation samples
predict_df = dataset.predict_df()   # Test samples + unlabeled samples

In [7]:
train_df.collect()

id,rings,et,eta,label,fold,is_test,is_val,is_train
u64,list[f32],f32,f32,bool,i64,bool,bool,bool
0,"[-2.705197, 9.096173, … -7.955918]",48871.601562,0.797343,false,1,true,false,true
3,"[-9.464378, 2.82857, … 5.509369]",7989.174316,0.683489,true,2,true,false,true
4,"[-2.379803, 8.889713, … -7.701864]",368.296509,0.296633,false,1,true,false,true
5,"[-9.447341, 2.652715, … 5.430638]",42932.371094,0.601509,true,2,true,false,true
6,"[-2.487935, 9.077365, … -7.879599]",32589.847656,0.166459,false,1,true,false,true
…,…,…,…,…,…,…,…,…
9993,"[-9.503527, 2.720485, … 5.366321]",32175.84375,0.445212,true,2,true,false,true
9995,"[-2.510789, 9.011859, … -7.815817]",37780.175781,0.139575,false,2,true,false,true
9997,"[-9.287633, 2.760057, … 5.751782]",8060.95459,0.474787,true,2,true,false,true
